In [4]:
############################################################
# DEGREE DISTRIBUTION + EXPONENTIAL FIT (R VERSION)
############################################################

library(tidyverse)

# =============================
# 1. File paths
# =============================
ms_edge_path  <- "/home/goel107/NCEMS/ANALYSIS/07_15_2025/The_Yeast_Interactome_edges.csv"
y2h_edge_path <- "/home/goel107/NCEMS/ANALYSIS/07_15_2025/Y2H_union-clean_edges.csv"

# =============================
# 2. Compute degree distribution (CORRECT)
#    Exactly matches Python logic:
#    degree_per_protein = table(all_nodes)
#    unique_degrees, counts = np.unique(degree_per_protein.values)
# =============================
compute_degree_distribution <- function(src, tgt) {
    all_nodes <- c(src, tgt)

    # Degree per protein
    degree_per_protein <- table(all_nodes)
    degree_values <- as.numeric(degree_per_protein)

    # Distribution of degree values
    degree_distribution <- table(degree_values)

    df <- data.frame(
        degree = as.numeric(names(degree_distribution)),
        count  = as.numeric(degree_distribution)
    )

    df
}

# =============================
# 3. Exponential fit y = a * exp(b*x)
# =============================
fit_exponential <- function(df) {
    df <- df %>% 
        filter(!is.na(degree), !is.na(count), degree > 0, count > 0)

    df <- df %>% mutate(log_count = log(count))

    fit <- lm(log_count ~ degree, data = df)

    a  <- exp(coef(fit)[1])
    b  <- coef(fit)[2]
    R2 <- summary(fit)$r.squared

    list(a = a, b = b, R2 = R2, df_fit = df)
}

# =============================
# 4. Load and process datasets
# =============================

### --- Mass Spec ---
df_ms_raw <- read.csv(ms_edge_path)
df_ms_deg <- compute_degree_distribution(df_ms_raw$source, df_ms_raw$target)
fit_ms <- fit_exponential(df_ms_deg)

### --- Y2H ---
df_y2h_raw <- read.csv(y2h_edge_path)

split_interaction <- function(s) strsplit(s, " \\(interacts with\\) ")[[1]]

pairs <- lapply(df_y2h_raw$name, split_interaction)
src <- sapply(pairs, `[`, 1)
tgt <- sapply(pairs, `[`, 2)

df_y2h_deg <- compute_degree_distribution(src, tgt)
fit_y2h <- fit_exponential(df_y2h_deg)

# =============================
# 5. Prepare fitted curves
# =============================
make_curve <- function(fit, dataset_name) {
    x_vals <- seq(min(fit$df_fit$degree), max(fit$df_fit$degree), length.out = 300)
    tibble(
        degree  = x_vals,
        count   = fit$a * exp(fit$b * x_vals),
        dataset = dataset_name
    )
}

curve_ms  <- make_curve(fit_ms,  "MS Interactome")
curve_y2h <- make_curve(fit_y2h, "Y2H Interactome")

df_plot <- bind_rows(
    df_ms_deg  %>% mutate(dataset = "MS Interactome"),
    df_y2h_deg %>% mutate(dataset = "Y2H Interactome")
)

curve_data <- bind_rows(curve_ms, curve_y2h)

# =============================
# 6. Plot (semi-log: log y-axis)
# =============================
p <- ggplot(df_plot, aes(x = degree, y = count, color = dataset)) +
    geom_point(size = 3, alpha = 0.9) +
    geom_line(data = curve_data,
              aes(x = degree, y = count, color = dataset),
              linetype = "dashed", linewidth = 1.1) +
    scale_y_log10() +
    theme_bw(base_size = 12, base_family = "Arial") +
    theme(
        panel.border = element_blank(),
        axis.line     = element_line(color = "black", linewidth = 1.1),
        panel.grid    = element_blank(),
        legend.title  = element_blank(),
        legend.position = "top"
    ) +
    xlab("Node degree") +
    ylab("Number of nodes") +
    scale_color_manual(values=c("MS Interactome"="#DDAA33",
                                "Y2H Interactome"="#BB5566")) +
    annotate("text", x=20, y=150,
             label=sprintf("MS: y = %.1f * e^(%.3f x)\nR² = %.4f",
                           fit_ms$a, fit_ms$b, fit_ms$R2),
             color="#DDAA33", size=4, fontface="bold") +
    annotate("text", x=20, y=50,
             label=sprintf("Y2H: y = %.1f * e^(%.3f x)\nR² = %.4f",
                           fit_y2h$a, fit_y2h$b, fit_y2h$R2),
             color="#BB5566", size=4, fontface="bold")

ggsave(
    "/home/goel107/NCEMS/ANALYSIS/07_15_2025/FIGURE1/1D-linear_plot(exponential-fit)_R.pdf",
    p, width = 4, height = 4, dpi = 300
)

print(p)

# =============================
# 7. Print numeric results
# =============================
cat(sprintf("MS:  a = %.4f,  b = %.4f,  R2 = %.4f\n",
            fit_ms$a, fit_ms$b, fit_ms$R2))
cat(sprintf("Y2H: a = %.4f,  b = %.4f,  R2 = %.4f\n",
            fit_y2h$a, fit_y2h$b, fit_y2h$R2))


Warning message in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)):
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)):
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)):
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)):
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)):
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)):
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)):
“font family 'Arial' not found in PostScript font database”
Warning message in grid.Call(C_stringMetric, as.graphicsAnnot(x$label

ERROR: Error in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, : invalid font type
